## SETUP

Este noteboook realiza a ingestão de dados e cria tabela no schema bronze




## Parte 1: Garantir que estamos usando o catálogo correto e que o schame 00_langing

In [0]:
select current_catalog(), current_schema()

In [0]:
USE CATALOG smart_claims_dev


In [0]:
select current_catalog(), current_schema()

In [0]:
USE SCHEMA 00_landing

In [0]:
select current_catalog(), current_schema()

## Parte 2: Explore os dados do arquivos

**COMANDO: LIST**

O que faz:
- Lista o caminho absolutor do arquivos presentes no volume
- LIsta o tamnho dos arquivos e quando foram modificados

In [0]:
list '/Volumes/smart_claims_dev/00_landing/sql_server'

Realizando o mesmo processo do  `List`, utizando o `Python`

In [0]:
%python
files = dbutils.fs.ls('/Volumes/smart_claims_dev/00_landing/sql_server')
display(files)

### Parte 3: Apresentação dos dados
Exibe o conteúdo do arquivo CSV, através do caminho do volume

In [0]:
select * 
from csv.`/Volumes/smart_claims_dev/00_landing/sql_server`

## Parte 4: Ingestão de Dados BATCH usando CTAs
CTAs com CREATE TABLES AS é uma maneira eficiente de ingerir dados em massa

Dentro do Unity Catalog, podemos criar tabelas Delta a partir de arquivos em volumes

In [0]:
select
  *
from
  read_files('/Volumes/smart_claims_dev/00_landing/sql_server', format => 'csv')
limit 10

### Ingestão de dados usando SQL
Cria tabela `claims` na camada bronze

In [0]:
DROP TABLE IF EXISTS smart_claims_dev.01_bronze.claims;

CREATE TABLE smart_claims_dev.01_bronze.claims AS 
SELECT
*
FROM read_files('/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv', format => 'csv')

In [0]:
select * from smart_claims_dev.01_bronze.claims

Descreve a estrutura da tabela  

In [0]:
describe table extended smart_claims_dev.01_bronze.claims

### Ingestão de dados usando Python

In [0]:
drop table if exists smart_claims_dev.01_bronze.claims

In [0]:
select * from smart_claims_dev.01_bronze.claims

In [0]:
%python
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load("/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv")
)

df.write.mode("overwrite").saveAsTable("smart_claims_dev.01_bronze.claims")

claims_table = spark.table("smart_claims_dev.01_bronze.claims")

display(claims_table)

### Ingestão de dados com COPY INTO

In [0]:
drop table if exists smart_claims_dev.`01_bronze`.claims;

create or replace table smart_claims_dev.`01_bronze`.claims (
    claim_no STRING,
    policy_no STRING,
    claim_date STRING,
    months_as_customer STRING,
    injury STRING,
    property STRING,
    vehicle STRING,
    total STRING,
    collision_type STRING,
    number_of_vehicles_involved STRING,
    age STRING,
    insured_relationship STRING,
    license_issue_date STRING,
    date STRING,        -- renomeei `date` para evitar conflito
    hour STRING,
    type STRING,
    severity STRING,
    number_of_witnesses STRING,
    suspicious_activity STRING
)
COMMENT 'Landing raw claims data (CSV → Bronze, raw strings)';

In [0]:
copy into smart_claims_dev.01_bronze.claims
from '/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false')

In [0]:
%python
df = spark.read.csv("/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv", header=True)
print(df.columns)

In [0]:
SELECT * FROM smart_claims_dev.01_bronze.claims